# SCBA Visualization
30 June 2026 \
Yong Da Li


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# data class to hold data across iterations
from scba_container.scba_container import SCBAContainer

from quatrex.core.config import parse_config
from pathlib import Path

# 20260402_150451_energy201_iter20_reduced_rank50
# 20260402_151203_energy201_iter20_reduced_rank100

# test of new meir-wingreen implementation in rgf solver
# 20260415_164308_energy2001_iter20_rank250

# check same settings as adaptive on meir-wingreen rebase
# 20260421_152051_energy2001_iter10_rank125


simulation_dir = Path("./../../examples/w90/carbon-nanotube/gw")
output_dir = simulation_dir / "outputs"
output_file_prefix = output_dir / "scba_variables"
archive_dir = simulation_dir / "archive" / "20260421_152051_energy2001_iter10_rank125"
archive_file_prefix = archive_dir / "scba_variables"

# change these to switch between 'output' and 'archive'
data_files = 'output'  # 'archive' or 'output'

# === user parameters ===
iteration_to_plot = 1
sample_idx_to_plot = 37

# region: === load data ===
if data_files == 'output':
    data_file_prefix = output_file_prefix
    data_dir = output_dir
    config_dir = simulation_dir
elif data_files == 'archive':
    data_file_prefix = archive_file_prefix
    data_dir = archive_dir
    config_dir = archive_dir
else:
    raise ValueError(f"Invalid data_files value: {data_files}. Must be 'output' or 'archive'.")

config = parse_config(config_dir / "quatrex_config.toml")

max_idx = 43823  # total nnz entries - 1, hardcoded for w90/carbon-nanotube/gw example
sample_indices = np.load(f"{data_file_prefix}_sample_indices.npy")
num_samples = len(sample_indices)

SCBADataObj = SCBAContainer(
    max_iterations=config.scba.max_iterations,
    energy_window_min=config.electron.energy_window_min,
    energy_window_max=config.electron.energy_window_max,
    energy_window_num=config.electron.energy_window_num,
    num_samples=num_samples,
)

SCBADataObj.load_sample_indices(data_file_prefix)
SCBADataObj.load_adaptive_grids(data_dir)

for i in range(config.scba.max_iterations):
    SCBADataObj.load_g_data(data_file_prefix, iteration=i)
    SCBADataObj.load_p_data(data_file_prefix, iteration=i)
    SCBADataObj.load_w_data(data_file_prefix, iteration=i)
    SCBADataObj.load_sigma_data(data_file_prefix, iteration=i)

assert (
    sample_idx_to_plot <= num_samples
), f"sample_idx_to_plot={sample_idx_to_plot} exceeds num_samples={num_samples}"
nnz_index_to_plot = SCBADataObj.sample_indices[sample_idx_to_plot]
# endregion

# region: === plotting ===
fig, axs = plt.subplots(4, 3, figsize=(10, 8), sharey='row')

SCBADataObj.plot_iteration(
    axs,
    iteration_to_plot,
    sample_idx_to_plot,
)

fig.suptitle(
    f"SCBA Data Visualization $N_E$={config.electron.energy_window_num} - Iteration {iteration_to_plot}, Sampled NNZ Index ({sample_idx_to_plot}/{num_samples}) {nnz_index_to_plot}/{max_idx} = {nnz_index_to_plot/max_idx:.2%}"
)
plt.tight_layout()
plt.show()
# endregion


AssertionError: sample_idx_to_plot=370 exceeds num_samples=100